In [1]:
import qewton
import math

X = qewton.Variable("x", 2)
T = qewton.Variable("t", 1)
D = qewton.Variable("D", 1)
U = qewton.Variable("U", 1)

In [2]:
w, h = 20.0, 20.0
T_end = 20.0

A_x = qewton.geometries.Rectangle(X, [0, 0], w, h)
A_t = qewton.geometries.Interval(T, 0, T_end)
A_D = qewton.geometries.Interval(D, 0.1, 1.0)

In [3]:
inner_sampler = qewton.RandomUniformSampler(A_x * A_t * A_D, n_points=35000)
initial_sampler = qewton.RandomUniformSampler(A_x*A_t.boundary_left*A_D, n_points=10000)
boundary_sampler = qewton.RandomUniformSampler(A_x.boundary*A_t*A_D, n_points=20000)

In [4]:
model = qewton.FCN(
    in_neurons=X*T*D,
    hidden_neurons=50,
    out_neurons=U,
    n_hidden_layers=3,
    activation=qewton.bb.Tanh,
)

In [7]:
def residual_fun(u: U, x: X, t : T, D: D):  # type: ignore
    return D*u.laplacian(x) - u.gradient(t)

pde_graph = qewton.PINNPipeline(inner_sampler, [model], residual=residual_fun)
pde_constraint = pde_graph.constraint

In [8]:
def boundary_fun(u: U):  # type: ignore
    return u

bc_graph = qewton.PINNPipeline(boundary_sampler, [model], residual=boundary_fun, 
                               residual_name="Boundary")
bc_constraint = bc_graph.constraint

In [9]:
def f(x):
    sin_x = qewton.bb.Sin()(math.pi/w * x[:, :1])
    sin_y = qewton.bb.Sin()(math.pi/h * x[:, 1:])
    return sin_x * sin_y

def initial_residual(u: U, x: X):  # type: ignore
    return u-f(x)

initial_graph = qewton.PINNPipeline(boundary_sampler, [model], residual=initial_residual, 
                               residual_name="Initial")
initial_constraint = initial_graph.constraint

In [10]:
adam_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.Adam(),
    lr=0.001,
    max_iterations=3000,
)

lbfgs_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.LBFGS(),
    lr=0.1,
    max_iterations=100,
    optimizer_args={"max_eval": 10},
)

fix_sampler_callback = qewton.optim.CacheDataCallback(
    data_nodes=[inner_sampler, initial_sampler, boundary_sampler],
    phase_to_start_cache=lbfgs_phase
)

trainer = qewton.optim.GraphBasedTrainer(
    optimization_phases=[adam_phase, lbfgs_phase],
    graphs=[pde_graph, initial_graph, bc_graph],
    callbacks=[fix_sampler_callback],
    training_objectives=[pde_constraint, initial_constraint, bc_constraint],
    device=qewton.cuda(0),
)

trainer.run()

Optimization Phase 2: 100%|██████████| 100/100 [00:04<00:00, 23.14it/s, loss=9.61e-6]
